# Tarefa 3: tres ficheiros cuadrimestrais

**Obxectivo:** calcular o importe total das vendas de 2025 e o número de vendas a partir de tres ficheiros, un por cuadrimestre.

Fases da resolución:

1. Preparación
2. Exploración dos ficheiros
3. Carga dos datos
4. Validación: buscar problemas
5. Limpeza: eliminar duplicados
6. Resultado

Os ficheiros deben estar descomprimidos na carpeta `datos/raw`.

## 1. Preparación

Importamos as librarías e definimos dúas funcións para mostrar números e euros en formato galego.

In [ ]:
from pathlib import Path
import pandas as pd

def eur(x):
    return f"{x:,.2f} €".replace(",", "X").replace(".", ",").replace("X", ".")

def num(x):
    return f"{x:,}".replace(",", ".")

carpeta = Path("../datos/raw")

## 2. Exploración dos ficheiros

Antes de cargar nada, miramos que ficheiros hai, canto ocupan e como son as primeiras liñas. Así decidimos como lelos (separador, decimais, codificación).

In [ ]:
ficheiros = sorted(carpeta.glob("vendas_2025_cuadrimestre*.csv"))
for f in ficheiros:
    print(f"{f.name:35} {f.stat().st_size / 1e6:6.1f} MB")
assert len(ficheiros) == 3, f"Esperábanse 3 ficheiros e atopáronse {len(ficheiros)}"

In [ ]:
def ver_inicio(ficheiro, n_liñas=3, encoding="utf-8"):
    """Mostra as primeiras liñas do ficheiro tal como están no disco, sen interpretar."""
    with open(ficheiro, encoding=encoding) as f:
        for _ in range(n_liñas):
            print(repr(f.readline()))

for f in ficheiros:
    print(f"--- {f.name}")
    ver_inicio(f)

**Observacións:** os tres ficheiros teñen a mesma estrutura que nas tarefas anteriores: separador `;`, coma decimal e codificación UTF-8. Hai unha columna `id_venda` que identifica cada venda.

## 3. Carga dos datos

Cargamos cada ficheiro e engadimos unha columna `orixe` para saber de onde vén cada fila. Será útil se atopamos problemas.

In [ ]:
partes = []
for f in ficheiros:
    df = pd.read_csv(f, sep=";", decimal=",", encoding="utf-8")
    df["orixe"] = f.stem.replace("vendas_2025_", "")
    partes.append(df)

vendas = pd.concat(partes, ignore_index=True)
print(f"Filas cargadas: {num(len(vendas))}")
vendas.head()

In [ ]:
# Os tipos de datos son os esperados? importe debe ser numérico
print(vendas.dtypes)
assert pd.api.types.is_numeric_dtype(vendas["importe"]), "importe non se leu como número"

## 4. Validación: buscar problemas

### 4.1. Resumo por ficheiro

Comprobamos cantas filas ten cada ficheiro e que rango de datas cobre.

In [ ]:
resumo = vendas.groupby("orixe").agg(
    filas=("id_venda", "size"),
    desde=("data", "min"),
    ata=("data", "max"),
    importe=("importe", "sum"),
)
resumo.assign(filas=resumo["filas"].map(num), importe=resumo["importe"].map(eur))

**Observación:** o cuadrimestre 2 comeza o 16 de abril, pero o cuadrimestre 1 remata o 30 de abril. As datas solápanse e iso suxire que pode haber vendas repetidas.

### 4.2. Identificadores repetidos

Se cada venda ten un `id_venda` único, non debería haber ningún repetido.

In [ ]:
print(f"Filas totais:       {num(len(vendas))}")
print(f"id_venda distintos: {num(vendas['id_venda'].nunique())}")

repetidos = vendas[vendas.duplicated("id_venda", keep=False)]
print(f"Filas con id_venda repetido: {num(len(repetidos))}")

In [ ]:
# En que ficheiros e datas están os repetidos?
repetidos.groupby("orixe")["data"].agg(["count", "min", "max"])

### 4.3. Son copias exactas?

Antes de borrar nada, comprobamos que as filas repetidas teñen os mesmos datos. Se houbese dúas vendas distintas co mesmo `id_venda`, borrar unha sería perder unha venda real.

In [ ]:
columnas = [c for c in vendas.columns if c != "orixe"]
distintas = repetidos.drop_duplicates(subset=columnas)["id_venda"].duplicated().sum()
print(f"id repetidos con datos diferentes: {distintas}")
assert distintas == 0, "Hai id repetidos con datos distintos: revisar á man"

In [ ]:
# Un exemplo: a mesma venda nos dous ficheiros
exemplo = repetidos["id_venda"].iloc[0]
vendas[vendas["id_venda"] == exemplo]

## 5. Limpeza: eliminar duplicados

Os repetidos son copias exactas, así que gardamos só a primeira aparición de cada `id_venda`.

In [ ]:
unicas = vendas.drop_duplicates("id_venda", keep="first")

print(f"Filas antes:  {num(len(vendas))}")
print(f"Filas despois: {num(len(unicas))}")
print(f"Eliminadas:   {num(len(vendas) - len(unicas))} filas, {eur(vendas['importe'].sum() - unicas['importe'].sum())}")

In [ ]:
# Comprobación: agora cada id_venda aparece unha soa vez
assert unicas["id_venda"].is_unique

## 6. Resultado

In [ ]:
print(f"Suma directa (incorrecta): {num(len(vendas))} vendas, {eur(vendas['importe'].sum())}")
print()
print(f"Número de vendas: {num(len(unicas))}")
print(f"Importe total:    {eur(unicas['importe'].sum())}")

**Conclusión:** a exportación do cuadrimestre 2 incluía tamén a segunda quincena de abril, que xa estaba no cuadrimestre 1. Sumar os ficheiros sen comprobar daría un total inflado. O `id_venda` permite detectalo e eliminar os duplicados con seguridade.